In [1]:
import serial
import time
from errors import TLE
from tools import Calculator
import threading

class controller:
    """Represent controller and its associated operations."""
    def __init__(self, serName, bps = 9600, timeout = 10):
        """Initialize controller dependencies and internal state."""
        self.serName = serName
        self.bps = bps
        self.timeout = timeout
        self.ser = serial.Serial(self.serName, self.bps, timeout = 0)
        
        self.spinThread = threading.Thread(target = self.blockSpin, args = (0, 2,))
        self.closeThread = threading.Thread(target = self.blockClose)
        self.processing = False

    def writeList(self, command):
        """Write list."""
        self.ser.write(bytes(command))
        time.sleep(0.2)

    def enable(self):
        """Enable."""
        command = [0x01, 0x06, 0x00, 0xB6, 0x00, 0x01, 0xA9, 0xEC]
        self.writeList(command)
        
    
    def setSpeed(self, speed):
        """Set speed."""
        command = [0x01, 0x06, 0x00, 0x56]
        command += Calculator.intToList(speed)
        command += Calculator.crc(bytes(command), flag = 'list')
        self.writeList(command)
    
    def start(self, flag = False):
        """Start."""
        command = [0x01, 0x06, 0x00, 0x66, 0x00, 0x01]
        if flag:
            command = [0x01, 0x06, 0x00, 0x66, 0x00, 0x02]
        command += Calculator.crc(bytes(command), flag = 'list')
        self.writeList(command)
        
    def stop(self):
        """Stop."""
        command = [0x01, 0x06, 0x00, 0x66, 0x00, 0x00, 0x69, 0xD5]
        self.writeList(command)

    def blockSpin(self, speed, spinTime): #without thread
        """Block spin."""
        while(self.processing == True):
            time.sleep(0.1)
        self.processing = True
        self.enable()
        self.setSpeed(speed)
        self.start()
        time.sleep(spinTime)
        self.stop()
        self.Processing = False
    
    def spin(self, speed, spinTime):
        """Spin."""
        self.spinThread = threading.Thread(target = self.blockSpin, args = (speed, spinTime,))
        self.spinThread.start()
    
    def blockClose(self):
        """Block close."""
        self.spinThread.join()
        self.ser.close()

    def close(self):
        """Close."""
        self.closeThread = threading.Thread(target = self.blockClose)
        self.closeThread.start()
        

if __name__ == '__main__':
    spinCoater = controller('COM10')
    try:
        spinCoater.spin(speed = 1000, spinTime = 2)
        spinCoater.close()
    except Exception as e:
        spinCoater.close()
        raise e